# MiniMind 从零训练 - Google Colab 版本

本 Notebook 专为 Google Colab 设计，可直接在 Colab 上运行。

## 快速开始
1. 打开此 Notebook 在 Google Colab 中
2. 选择运行时类型：**GPU** (Runtime > Change runtime type > Hardware accelerator > GPU)
3. 按顺序运行每个单元格

## 项目信息
- 原始项目: [minimind](https://github.com/jingyaogong/minimind)
- 模型规模: ~64M 参数
- 推荐 Colab GPU: T4 (免费) 或 A100 (Pro)

## 第一部分：环境准备

In [ ]:
# 检查 GPU 可用性
import torch
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 设备: {torch.cuda.get_device_name(0)}")
    print(f"GPU 显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ 未检测到 GPU，请更改 Colab 运行时类型为 GPU")

# 设置训练设备
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用设备: {device}")

In [ ]:
# 安装必要的依赖
!pip install transformers datasets accelerate -q
print("依赖安装完成！")

In [ ]:
# 下载 tokenizer 配置文件（模型结构已在 Notebook 中实现）
!mkdir -p ./minimind/model
!wget -q -O ./minimind/model/tokenizer.json https://huggingface.co/jingyaogong/minimind-3/resolve/main/tokenizer.json
!wget -q -O ./minimind/model/tokenizer_config.json https://huggingface.co/jingyaogong/minimind-3/resolve/main/tokenizer_config.json
print("Tokenizer 配置文件下载完成！")

In [ ]:
# 导入所有需要的模块
import os
import sys
import math
import json
import random
import time
import warnings
import __main__
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, PreTrainedModel, GenerationMixin, PretrainedConfig
from transformers.activations import ACT2FN
from transformers.modeling_outputs import MoeCausalLMOutputWithPast
from datasets import load_dataset, Features, Sequence, Value

import matplotlib.pyplot as plt
from IPython.display import clear_output

warnings.filterwarnings('ignore')
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 兼容新版 transformers：部分内部能力探测会尝试读取模型类所在源码文件。
# 在 Notebook / Colab 中，自定义模型通常定义在 __main__，可能没有可读取的源码文件，
# 这里将异常降级为 False，表示不启用对应的可选实现，而不是直接初始化失败。
def _patch_transformers_source_probe(method_name):
    method = getattr(PreTrainedModel, method_name, None)
    if method is None:
        return

    func = method.__func__ if hasattr(method, '__func__') else method

    def _safe_probe(cls):
        try:
            return func(cls)
        except (AttributeError, FileNotFoundError, OSError, TypeError):
            return False

    setattr(PreTrainedModel, method_name, classmethod(_safe_probe))

_patch_transformers_source_probe('_can_set_attn_implementation')
_patch_transformers_source_probe('_can_set_experts_implementation')

print("所有依赖导入完成！")

In [ ]:
# 创建必要的目录结构
os.makedirs('out', exist_ok=True)
os.makedirs('dataset', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

# 设置路径（指向克隆的仓库）
MODEL_DIR = './minimind/model'
DATASET_DIR = './dataset'
OUTPUT_DIR = './out'
CHECKPOINT_DIR = './checkpoints'

print(f"目录结构已创建")
print(f"模型目录: {MODEL_DIR}")
print(f"数据目录: {DATASET_DIR}")
print(f"输出目录: {OUTPUT_DIR}")

## 第二部分：模型架构定义

In [ ]:
# MiniMind 模型配置
class MiniMindConfig(PretrainedConfig):
    model_type = "minimind"
    
    def __init__(self, hidden_size=768, num_hidden_layers=8, use_moe=False, **kwargs):
        super().__init__(**kwargs)
        self.hidden_size = hidden_size
        self.num_hidden_layers = num_hidden_layers
        self.use_moe = use_moe
        self.dropout = kwargs.get("dropout", 0.0)
        self.vocab_size = kwargs.get("vocab_size", 6400)
        self.bos_token_id = kwargs.get("bos_token_id", 1)
        self.eos_token_id = kwargs.get("eos_token_id", 2)
        self.flash_attn = kwargs.get("flash_attn", True)
        self.num_attention_heads = kwargs.get("num_attention_heads", 8)
        self.num_key_value_heads = kwargs.get("num_key_value_heads", 4)
        self.head_dim = kwargs.get("head_dim", self.hidden_size // self.num_attention_heads)
        self.hidden_act = kwargs.get("hidden_act", 'silu')
        self.intermediate_size = kwargs.get("intermediate_size", math.ceil(hidden_size * math.pi / 64) * 64)
        self.max_position_embeddings = kwargs.get("max_position_embeddings", 32768)
        self.rms_norm_eps = kwargs.get("rms_norm_eps", 1e-6)
        self.rope_theta = kwargs.get("rope_theta", 1e6)
        self.inference_rope_scaling = kwargs.get("inference_rope_scaling", False)
        self.rope_scaling = {
            "beta_fast": 32,
            "beta_slow": 1,
            "factor": 16,
            "original_max_position_embeddings": 2048,
            "attention_factor": 1.0,
            "type": "yarn"
        } if self.inference_rope_scaling else None
        # MoE 配置
        self.num_experts = kwargs.get("num_experts", 4)
        self.num_experts_per_tok = kwargs.get("num_experts_per_tok", 1)
        self.moe_intermediate_size = kwargs.get("moe_intermediate_size", self.intermediate_size)
        self.norm_topk_prob = kwargs.get("norm_topk_prob", True)
        self.router_aux_loss_coef = kwargs.get("router_aux_loss_coef", 5e-4)

print("MiniMindConfig 定义完成！")

In [ ]:
# RMSNorm 层
class RMSNorm(torch.nn.Module):
    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        return (self.weight * self.norm(x.float())).type_as(x)

print("RMSNorm 定义完成！")

In [ ]:
# RoPE 位置编码
def precompute_freqs_cis(dim: int, end: int = int(32 * 1024), rope_base: float = 1e6, rope_scaling: dict = None):
    freqs, attn_factor = 1.0 / (rope_base ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim)), 1.0
    if rope_scaling is not None:  # YaRN
        orig_max, factor, beta_fast, beta_slow, attn_factor = (
            rope_scaling.get("original_max_position_embeddings", 2048), rope_scaling.get("factor", 16),
            rope_scaling.get("beta_fast", 32.0), rope_scaling.get("beta_slow", 1.0), rope_scaling.get("attention_factor", 1.0)
        )
        if end / orig_max > 1.0:
            inv_dim = lambda b: (dim * math.log(orig_max / (b * 2 * math.pi))) / (2 * math.log(rope_base))
            low, high = max(math.floor(inv_dim(beta_fast)), 0), min(math.ceil(inv_dim(beta_slow)), dim // 2 - 1)
            ramp = torch.clamp((torch.arange(dim // 2, device=freqs.device).float() - low) / max(high - low, 0.001), 0, 1)
            freqs = freqs * (1 - ramp + ramp / factor)
    t = torch.arange(end, device=freqs.device)
    freqs = torch.outer(t, freqs).float()
    freqs_cos = torch.cat([torch.cos(freqs), torch.cos(freqs)], dim=-1) * attn_factor
    freqs_sin = torch.cat([torch.sin(freqs), torch.sin(freqs)], dim=-1) * attn_factor
    return freqs_cos, freqs_sin

def apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1):
    def rotate_half(x): return torch.cat((-x[..., x.shape[-1] // 2:], x[..., : x.shape[-1] // 2]), dim=-1)
    q_embed = ((q * cos.unsqueeze(unsqueeze_dim)) + (rotate_half(q) * sin.unsqueeze(unsqueeze_dim))).to(q.dtype)
    k_embed = ((k * cos.unsqueeze(unsqueeze_dim)) + (rotate_half(k) * sin.unsqueeze(unsqueeze_dim))).to(k.dtype)
    return q_embed, k_embed

def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    bs, slen, num_key_value_heads, head_dim = x.shape
    if n_rep == 1: return x
    return (x[:, :, :, None, :].expand(bs, slen, num_key_value_heads, n_rep, head_dim).reshape(bs, slen, num_key_value_heads * n_rep, head_dim))

print("RoPE 位置编码定义完成！")

In [ ]:
# Attention 机制 (GQA)
class Attention(nn.Module):
    def __init__(self, config: MiniMindConfig):
        super().__init__()
        self.num_key_value_heads = config.num_attention_heads if config.num_key_value_heads is None else config.num_key_value_heads
        self.n_local_heads = config.num_attention_heads
        self.n_local_kv_heads = self.num_key_value_heads
        self.n_rep = self.n_local_heads // self.n_local_kv_heads
        self.head_dim = config.head_dim
        self.q_proj = nn.Linear(config.hidden_size, config.num_attention_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.hidden_size, self.num_key_value_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.hidden_size, self.num_key_value_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(config.num_attention_heads * self.head_dim, config.hidden_size, bias=False)
        self.q_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)
        self.k_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.dropout = config.dropout
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention') and config.flash_attn

    def forward(self, x, position_embeddings, past_key_value=None, use_cache=False, attention_mask=None):
        bsz, seq_len, _ = x.shape
        xq, xk, xv = self.q_proj(x), self.k_proj(x), self.v_proj(x)
        xq = xq.view(bsz, seq_len, self.n_local_heads, self.head_dim)
        xk = xk.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)
        xv = xv.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)
        xq, xk = self.q_norm(xq), self.k_norm(xk)
        cos, sin = position_embeddings
        xq, xk = apply_rotary_pos_emb(xq, xk, cos, sin)
        if past_key_value is not None:
            xk = torch.cat([past_key_value[0], xk], dim=1)
            xv = torch.cat([past_key_value[1], xv], dim=1)
        past_kv = (xk, xv) if use_cache else None
        xq, xk, xv = (xq.transpose(1, 2), repeat_kv(xk, self.n_rep).transpose(1, 2), repeat_kv(xv, self.n_rep).transpose(1, 2))
        if self.flash and (seq_len > 1) and (past_key_value is None) and (attention_mask is None or torch.all(attention_mask == 1)):
            output = F.scaled_dot_product_attention(xq, xk, xv, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        else:
            scores = (xq @ xk.transpose(-2, -1)) / math.sqrt(self.head_dim)
            scores[:, :, :, -seq_len:] += torch.full((seq_len, seq_len), float("-inf"), device=scores.device).triu(1)
            if attention_mask is not None: scores += (1.0 - attention_mask.unsqueeze(1).unsqueeze(2)) * -1e9
            output = self.attn_dropout(F.softmax(scores.float(), dim=-1).type_as(xq)) @ xv
        output = output.transpose(1, 2).reshape(bsz, seq_len, -1)
        output = self.resid_dropout(self.o_proj(output))
        return output, past_kv

print("Attention 定义完成！")

In [ ]:
# FeedForward 层 (SwiGLU)
class FeedForward(nn.Module):
    def __init__(self, config: MiniMindConfig, intermediate_size: int = None):
        super().__init__()
        intermediate_size = intermediate_size or config.intermediate_size
        self.gate_proj = nn.Linear(config.hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, config.hidden_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, intermediate_size, bias=False)
        self.act_fn = ACT2FN[config.hidden_act]

    def forward(self, x):
        return self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))

# MoE 前馈网络 (可选)
class MOEFeedForward(nn.Module):
    def __init__(self, config: MiniMindConfig):
        super().__init__()
        self.config = config
        self.gate = nn.Linear(config.hidden_size, config.num_experts, bias=False)
        self.experts = nn.ModuleList([FeedForward(config, intermediate_size=config.moe_intermediate_size) for _ in range(config.num_experts)])
        self.act_fn = ACT2FN[config.hidden_act]

    def forward(self, x):
        batch_size, seq_len, hidden_dim = x.shape
        x_flat = x.view(-1, hidden_dim)
        scores = F.softmax(self.gate(x_flat), dim=-1)
        topk_weight, topk_idx = torch.topk(scores, k=self.config.num_experts_per_tok, dim=-1, sorted=False)
        if self.config.norm_topk_prob: topk_weight = topk_weight / (topk_weight.sum(dim=-1, keepdim=True) + 1e-20)
        y = torch.zeros_like(x_flat)
        for i, expert in enumerate(self.experts):
            mask = (topk_idx == i)
            if mask.any():
                token_idx = mask.any(dim=-1).nonzero().flatten()
                weight = topk_weight[mask].view(-1, 1)
                y.index_add_(0, token_idx, (expert(x_flat[token_idx]) * weight).to(y.dtype))
            elif self.training:
                y[0, 0] += 0 * sum(p.sum() for p in expert.parameters())
        if self.training and self.config.router_aux_loss_coef > 0:
            load = F.one_hot(topk_idx, self.config.num_experts).float().mean(0)
            self.aux_loss = (load * scores.mean(0)).sum() * self.config.num_experts * self.config.router_aux_loss_coef
        else:
            self.aux_loss = scores.new_zeros(1).squeeze()
        return y.view(batch_size, seq_len, hidden_dim)

print("FeedForward 和 MOEFeedForward 定义完成！")

In [ ]:
# MiniMindBlock (Transformer Block)
class MiniMindBlock(nn.Module):
    def __init__(self, layer_id: int, config: MiniMindConfig):
        super().__init__()
        self.self_attn = Attention(config)
        self.input_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attention_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.mlp = FeedForward(config) if not config.use_moe else MOEFeedForward(config)

    def forward(self, hidden_states, position_embeddings, past_key_value=None, use_cache=False, attention_mask=None):
        residual = hidden_states
        hidden_states, present_key_value = self.self_attn(
            self.input_layernorm(hidden_states), position_embeddings,
            past_key_value, use_cache, attention_mask
        )
        hidden_states += residual
        hidden_states = hidden_states + self.mlp(self.post_attention_layernorm(hidden_states))
        return hidden_states, present_key_value

print("MiniMindBlock 定义完成！")

In [ ]:
# MiniMindModel (完整的 Transformer 堆叠)
class MiniMindModel(nn.Module):
    def __init__(self, config: MiniMindConfig):
        super().__init__()
        self.config = config
        self.vocab_size, self.num_hidden_layers = config.vocab_size, config.num_hidden_layers
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.dropout = nn.Dropout(config.dropout)
        self.layers = nn.ModuleList([MiniMindBlock(l, config) for l in range(self.num_hidden_layers)])
        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        freqs_cos, freqs_sin = precompute_freqs_cis(dim=config.head_dim, end=config.max_position_embeddings, rope_base=config.rope_theta, rope_scaling=config.rope_scaling)
        self.register_buffer("freqs_cos", freqs_cos, persistent=False)
        self.register_buffer("freqs_sin", freqs_sin, persistent=False)

    def forward(self, input_ids, attention_mask=None, past_key_values=None, use_cache=False, **kwargs):
        batch_size, seq_length = input_ids.shape
        if hasattr(past_key_values, 'layers'): past_key_values = None
        past_key_values = past_key_values or [None] * len(self.layers)
        start_pos = past_key_values[0][0].shape[1] if past_key_values[0] is not None else 0
        hidden_states = self.dropout(self.embed_tokens(input_ids))
        position_embeddings = (self.freqs_cos[start_pos:start_pos + seq_length], self.freqs_sin[start_pos:start_pos + seq_length])
        presents = []
        for layer, past_key_value in zip(self.layers, past_key_values):
            hidden_states, present = layer(
                hidden_states,
                position_embeddings,
                past_key_value=past_key_value,
                use_cache=use_cache,
                attention_mask=attention_mask
            )
            presents.append(present)
        hidden_states = self.norm(hidden_states)
        aux_loss = sum([l.mlp.aux_loss for l in self.layers if isinstance(l.mlp, MOEFeedForward)], hidden_states.new_zeros(1).squeeze())
        return hidden_states, presents, aux_loss

print("MiniMindModel 定义完成！")

In [ ]:
# MiniMindForCausalLM (因果语言模型)
class MiniMindForCausalLM(PreTrainedModel, GenerationMixin):
    config_class = MiniMindConfig
    
    def __init__(self, config: MiniMindConfig = None):
        self.config = config or MiniMindConfig()
        super().__init__(self.config)
        self.model = MiniMindModel(self.config)
        self.lm_head = nn.Linear(self.config.hidden_size, self.config.vocab_size, bias=False)
        self.model.embed_tokens.weight = self.lm_head.weight
    
    def forward(self, input_ids, attention_mask=None, past_key_values=None, use_cache=False, logits_to_keep=0, labels=None, **kwargs):
        hidden_states, past_key_values, aux_loss = self.model(input_ids, attention_mask, past_key_values, use_cache, **kwargs)
        slice_indices = slice(-logits_to_keep, None) if isinstance(logits_to_keep, int) else logits_to_keep
        logits = self.lm_head(hidden_states[:, slice_indices, :])
        loss = None
        if labels is not None:
            x, y = logits[..., :-1, :].contiguous(), labels[..., 1:].contiguous()
            loss = F.cross_entropy(x.view(-1, x.size(-1)), y.view(-1), ignore_index=-100)
        return MoeCausalLMOutputWithPast(loss=loss, aux_loss=aux_loss, logits=logits, past_key_values=past_key_values, hidden_states=hidden_states)
    
    @torch.inference_mode()
    def generate(self, inputs=None, attention_mask=None, max_new_tokens=8192, temperature=0.85, top_p=0.85, top_k=50, eos_token_id=2, streamer=None, use_cache=True, num_return_sequences=1, do_sample=True, repetition_penalty=1.0, **kwargs):
        input_ids = kwargs.pop("input_ids", inputs).repeat(num_return_sequences, 1)
        attention_mask = attention_mask.repeat(num_return_sequences, 1) if attention_mask is not None else None
        past_key_values = kwargs.pop("past_key_values", None)
        finished = torch.zeros(input_ids.shape[0], dtype=torch.bool, device=input_ids.device)
        if streamer: streamer.put(input_ids.cpu())
        for _ in range(max_new_tokens):
            past_len = past_key_values[0][0].shape[1] if past_key_values else 0
            outputs = self.forward(input_ids[:, past_len:], attention_mask, past_key_values, use_cache=use_cache, **kwargs)
            attention_mask = torch.cat([attention_mask, attention_mask.new_ones(attention_mask.shape[0], 1)], -1) if attention_mask is not None else None
            logits = outputs.logits[:, -1, :] / temperature
            if repetition_penalty != 1.0:
                for i in range(input_ids.shape[0]): logits[i, torch.unique(input_ids[i])] /= repetition_penalty
            if top_k > 0: 
                logits[logits < torch.topk(logits, top_k)[0][..., -1, None]] = -float('inf')
            if top_p < 1.0:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                mask = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1) > top_p
                mask[..., 1:], mask[..., 0] = mask[..., :-1].clone(), 0
                logits[mask.scatter(1, sorted_indices, mask)] = -float('inf')
            next_token = torch.multinomial(torch.softmax(logits, dim=-1), num_samples=1) if do_sample else torch.argmax(logits, dim=-1, keepdim=True)
            if eos_token_id is not None: next_token = torch.where(finished.unsqueeze(-1), next_token.new_full((next_token.shape[0], 1), eos_token_id), next_token)
            input_ids = torch.cat([input_ids, next_token], dim=-1)
            past_key_values = outputs.past_key_values if use_cache else None
            if streamer: streamer.put(next_token.cpu())
            if eos_token_id is not None:
                finished |= next_token.squeeze(-1).eq(eos_token_id)
                if finished.all(): break
        if streamer: streamer.end()
        if kwargs.get("return_kv"): return {'generated_ids': input_ids, 'past_kv': past_key_values}
        return input_ids

print("MiniMindForCausalLM 定义完成！")

## 第三部分：工具函数

In [ ]:
# 随机种子设置
def setup_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# 学习率调度
def get_lr(current_step, total_steps, lr):
    return lr * (0.1 + 0.45 * (1 + math.cos(math.pi * current_step / total_steps)))

# 日志输出
def Logger(content):
    print(content)

# 模型参数量统计
def get_model_params(model, config):
    total = sum(p.numel() for p in model.parameters()) / 1e6
    n_routed = getattr(config, 'n_routed_experts', getattr(config, 'num_experts', 0))
    n_active = getattr(config, 'num_experts_per_tok', 0)
    n_shared = getattr(config, 'n_shared_experts', 0)
    expert = sum(p.numel() for n, p in model.named_parameters() if 'mlp.experts.0.' in n) / 1e6
    shared_expert = sum(p.numel() for n, p in model.named_parameters() if 'mlp.shared_experts.0.' in n) / 1e6
    base = total - (expert * n_routed) - (shared_expert * n_shared)
    active = base + (expert * n_active) + (shared_expert * n_shared)
    if active < total: Logger(f'Model Params: {total:.2f}M-A{active:.2f}M')
    else: Logger(f'Model Params: {total:.2f}M')

print("工具函数定义完成！")

## 第四部分：数据集类

In [ ]:
# 数据预处理辅助函数
def pre_processing_chat(conversations, add_system_ratio=0.2):
    SYSTEM_PROMPTS = [
        "你是一个知识丰富的AI，尽力为用户提供准确的信息。",
        "你是minimind，一个小巧但有用的语言模型。",
        "你是一个专业的AI助手，请提供有价值的回答。",
    ]
    if any(conv.get('tools') for conv in conversations): return conversations
    if conversations[0].get('role') != 'system':
        if random.random() < add_system_ratio:
            return [{'role': 'system', 'content': random.choice(SYSTEM_PROMPTS)}] + conversations
    return conversations

def post_processing_chat(prompt_content, empty_think_ratio=0.2):
    if '<think>\n\n</think>\n\n' in prompt_content and random.random() > empty_think_ratio:
        prompt_content = prompt_content.replace('<think>\n\n</think>\n\n', '')
    return prompt_content

print("数据预处理函数定义完成！")

In [ ]:
# 预训练数据集
class PretrainDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_length=512):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = load_dataset('json', data_files=data_path, split='train')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        tokens = self.tokenizer(str(sample['text']), add_special_tokens=False, max_length=self.max_length - 2, truncation=True).input_ids
        tokens = [self.tokenizer.bos_token_id] + tokens + [self.tokenizer.eos_token_id]
        input_ids = tokens + [self.tokenizer.pad_token_id] * (self.max_length - len(tokens))
        input_ids = torch.tensor(input_ids, dtype=torch.long)
        labels = input_ids.clone()
        labels[input_ids == self.tokenizer.pad_token_id] = -100
        return input_ids, labels

print("PretrainDataset 定义完成！")

In [ ]:
# 指令微调数据集
class SFTDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, max_length=1024):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_length = max_length
        features = Features({
            'conversations': Sequence({
                'role': Value('string'),
                'content': Value('string'),
                'reasoning_content': Value('string'),
                'tools': Value('string'),
                'tool_calls': Value('string')
            })
        })
        try:
            self.samples = load_dataset('json', data_files=jsonl_path, split='train', features=features)
        except Exception as e:
            print(f"⚠️ datasets 解析 SFT 数据失败，回退到手动读取 JSONL: {e}")
            self.samples = self._load_jsonl_samples(jsonl_path)
        self.bos_id = tokenizer(f'{tokenizer.bos_token}assistant\n', add_special_tokens=False).input_ids
        self.eos_id = tokenizer(f'{tokenizer.eos_token}\n', add_special_tokens=False).input_ids

    def _normalize_message(self, message):
        if not isinstance(message, dict):
            return None

        tools = message.get('tools')
        if isinstance(tools, (dict, list)):
            tools = json.dumps(tools, ensure_ascii=False)
        elif tools is None:
            tools = ''
        else:
            tools = str(tools)

        tool_calls = message.get('tool_calls')
        if isinstance(tool_calls, (dict, list)):
            tool_calls = json.dumps(tool_calls, ensure_ascii=False)
        elif tool_calls is None:
            tool_calls = ''
        else:
            tool_calls = str(tool_calls)

        return {
            'role': str(message.get('role', '')),
            'content': '' if message.get('content') is None else str(message.get('content')),
            'reasoning_content': '' if message.get('reasoning_content') is None else str(message.get('reasoning_content')),
            'tools': tools,
            'tool_calls': tool_calls,
        }

    def _load_jsonl_samples(self, jsonl_path):
        samples = []
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line_no, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    sample = json.loads(line)
                except json.JSONDecodeError as e:
                    print(f"⚠️ 第 {line_no} 行 JSON 解析失败，已跳过: {e}")
                    continue

                conversations = sample.get('conversations') or sample.get('messages') or []
                normalized_conversations = []
                for message in conversations:
                    normalized = self._normalize_message(message)
                    if normalized is not None:
                        normalized_conversations.append(normalized)

                if normalized_conversations:
                    samples.append({'conversations': normalized_conversations})
        return samples

    def __len__(self):
        return len(self.samples)

    def create_chat_prompt(self, conversations):
        messages = []
        tools = None
        for message in conversations:
            message = dict(message)
            if message.get("role") == "system" and message.get("tools"):
                tools = json.loads(message["tools"]) if isinstance(message["tools"], str) else message["tools"]
            if message.get("tool_calls") and isinstance(message["tool_calls"], str):
                message["tool_calls"] = json.loads(message["tool_calls"])
            messages.append(message)
        return self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
            tools=tools
        )

    def generate_labels(self, input_ids):
        labels = [-100] * len(input_ids)
        i = 0
        while i < len(input_ids):
            if input_ids[i:i + len(self.bos_id)] == self.bos_id:
                start = i + len(self.bos_id)
                end = start
                while end < len(input_ids):
                    if input_ids[end:end + len(self.eos_id)] == self.eos_id:
                        break
                    end += 1
                for j in range(start, min(end + len(self.eos_id), self.max_length)):
                    labels[j] = input_ids[j]
                i = end + len(self.eos_id) if end < len(input_ids) else len(input_ids)
            else:
                i += 1
        return labels

    def __getitem__(self, index):
        sample = self.samples[index]
        conversations = pre_processing_chat(sample['conversations'])
        prompt = self.create_chat_prompt(conversations)
        prompt = post_processing_chat(prompt)
        input_ids = self.tokenizer(prompt).input_ids[:self.max_length]
        input_ids += [self.tokenizer.pad_token_id] * (self.max_length - len(input_ids))
        labels = self.generate_labels(input_ids)
        return torch.tensor(input_ids, dtype=torch.long), torch.tensor(labels, dtype=torch.long)

print("SFTDataset 定义完成！")

## 第五部分：数据准备（Colab 专用）

### 5.1 数据集说明

MiniMind 训练需要两种数据集：

**预训练数据 (Pretrain)**:
- 格式: `jsonl` 文件，每行一个 `{"text": "内容"}` 对象
- 用途: 让模型学习语言模式
- 推荐: `pretrain_t2t_mini.jsonl`（约 1.2GB）

**指令微调数据 (SFT)**:
- 格式: `jsonl` 文件，每行一个对话样本
- 用途: 让模型学会对话和遵循指令
- 推荐: `sft_t2t_mini.jsonl`（约 200MB）

数据来源: [HuggingFace - minimind_dataset](https://huggingface.co/datasets/jingyaogong/minimind_dataset)

### 5.2 方式一：从 HuggingFace 自动下载（推荐）

In [ ]:
# 从 HuggingFace 下载数据集
print("正在下载数据集...")

# 安装下载工具
!pip install huggingface_hub -q

from huggingface_hub import hf_hub_download

# 创建数据目录
os.makedirs('./dataset', exist_ok=True)

# 下载预训练数据
try:
    pretrain_path = hf_hub_download(
        repo_id="jingyaogong/minimind_dataset",
        repo_type="dataset",
        filename="pretrain_t2t_mini.jsonl",
        local_dir="./dataset"
    )
    print(f"✅ 预训练数据下载完成")
except Exception as e:
    print(f"⚠️ 预训练数据下载失败: {e}")

# 下载 SFT 数据
try:
    sft_path = hf_hub_download(
        repo_id="jingyaogong/minimind_dataset",
        repo_type="dataset",
        filename="sft_t2t_mini.jsonl",
        local_dir="./dataset"
    )
    print(f"✅ SFT 数据下载完成")
except Exception as e:
    print(f"⚠️ SFT 数据下载失败: {e}")

# 检查数据文件
print("\n数据文件检查:")
for f in ['pretrain_t2t_mini.jsonl', 'sft_t2t_mini.jsonl']:
    path = os.path.join('./dataset', f)
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024 / 1024
        print(f"  ✅ {f}: {size:.1f} MB")
    else:
        print(f"  ❌ {f}: 不存在")

### 5.3 方式二：手动上传数据

如果自动下载失败，可以手动上传数据：

1. 从 [HuggingFace](https://huggingface.co/datasets/jingyaogong/minimind_dataset) 下载所需文件
2. 在 Colab 左侧文件浏览器中，上传到 `./dataset/` 目录
3. 或者使用以下代码上传：

```python
from google.colab import files
os.makedirs('./dataset', exist_ok=True)
uploaded = files.upload()  # 选择文件上传
for filename in uploaded.keys():
    !mv {filename} ./dataset/
```

### 5.4 方式三：使用自定义数据

你也可以使用自己的数据进行训练：

**预训练数据格式** (`my_pretrain.jsonl`):
```jsonl
{"text": "这是一段预训练文本。"}
{"text": "这是另一段文本。"}
```

**SFT 数据格式** (`my_sft.jsonl`):
```jsonl
{"conversations": [{"role": "user", "content": "你好"}, {"role": "assistant", "content": "你好！"}]}
{"conversations": [{"role": "user", "content": "再见"}, {"role": "assistant", "content": "再见！"}]}
```

创建自定义数据后，修改训练配置中的 `data_path` 指向你的文件。

In [ ]:
# 创建示例数据（用于测试，如果没有下载真实数据）
import json

os.makedirs('./dataset', exist_ok=True)

# 检查是否已有数据文件
pretrain_exists = os.path.exists('./dataset/pretrain_t2t_mini.jsonl')
sft_exists = os.path.exists('./dataset/sft_t2t_mini.jsonl')

if not pretrain_exists:
    print("创建示例预训练数据（仅用于测试）...")
    with open('./dataset/pretrain_t2t_mini.jsonl', 'w', encoding='utf-8') as f:
        for i in range(100):
            f.write(json.dumps({"text": f"这是第{i+1}条预训练文本示例。"}, ensure_ascii=False) + '\n')
    print("✅ 示例预训练数据创建完成")

if not sft_exists:
    print("创建示例 SFT 数据（仅用于测试）...")
    with open('./dataset/sft_t2t_mini.jsonl', 'w', encoding='utf-8') as f:
        for i in range(50):
            sample = {
                "conversations": [
                    {"role": "user", "content": f"问题 {i+1}"},
                    {"role": "assistant", "content": f"回答 {i+1}"}
                ]
            }
            f.write(json.dumps(sample, ensure_ascii=False) + '\n')
    print("✅ 示例 SFT 数据创建完成")

if pretrain_exists and sft_exists:
    print("✅ 真实数据文件已存在，无需创建示例数据")

# 最终检查
print("\n数据文件状态:")
for f in ['pretrain_t2t_mini.jsonl', 'sft_t2t_mini.jsonl']:
    path = os.path.join('./dataset', f)
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024
        print(f"  ✅ {f}: {size:.1f} KB")
    else:
        print(f"  ❌ {f}: 不存在")

## 第六部分：预训练 (Pretrain)

In [ ]:
# 预训练配置（针对 Colab T4 GPU 优化）
pretrain_config = {
    'epochs': 2,
    'batch_size': 16,  # Colab T4 显存限制
    'learning_rate': 5e-4,
    'max_seq_len': 256,  # 减小序列长度节省显存
    'hidden_size': 768,
    'num_hidden_layers': 8,
    'use_moe': False,
    'accumulation_steps': 16,  # 增加梯度累积
    'grad_clip': 1.0,
    'log_interval': 20,
    'save_interval': 200,
    'num_workers': 0,
    'data_path': os.path.join(DATASET_DIR, 'pretrain_t2t_mini.jsonl'),
    'save_weight': 'pretrain',
}

# 检查数据文件是否存在
if not os.path.exists(pretrain_config['data_path']):
    print(f"⚠️ 数据文件不存在: {pretrain_config['data_path']}")
    print("请先运行数据下载单元格")
else:
    print("✅ 数据文件存在，可以开始训练！")

print(f"\n预训练配置 (Colab 优化版):")
for k, v in pretrain_config.items():
    print(f"  {k}: {v}")

In [ ]:
# 初始化预训练环境
setup_seed(42)

# 创建模型配置
lm_config = MiniMindConfig(
    hidden_size=pretrain_config['hidden_size'],
    num_hidden_layers=pretrain_config['num_hidden_layers'],
    use_moe=pretrain_config['use_moe']
)

# 加载 Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

# 创建模型
model = MiniMindForCausalLM(lm_config)
get_model_params(model, lm_config)
model = model.to(device)

# 创建数据集和数据加载器
train_ds = PretrainDataset(pretrain_config['data_path'], tokenizer, max_length=pretrain_config['max_seq_len'])
train_loader = DataLoader(train_ds, batch_size=pretrain_config['batch_size'], shuffle=True, num_workers=pretrain_config['num_workers'])

# 优化器和混合精度
optimizer = optim.AdamW(model.parameters(), lr=pretrain_config['learning_rate'])
scaler = torch.cuda.amp.GradScaler(enabled=True)

print("预训练环境初始化完成！")
print(f"训练样本数: {len(train_ds)}")
print(f"每个 epoch 步数: {len(train_loader)}")

In [ ]:
# 训练损失记录
loss_history = []
step_count = 0

# 预训练循环
def train_pretrain():
    global step_count, loss_history
    model.train()
    
    for epoch in range(pretrain_config['epochs']):
        Logger(f"\n{'='*50}")
        Logger(f"Epoch [{epoch + 1}/{pretrain_config['epochs']}]")
        Logger(f"{'='*50}")
        
        epoch_loss = 0
        start_time = time.time()
        
        for step, (input_ids, labels) in enumerate(train_loader):
            input_ids = input_ids.to(device)
            labels = labels.to(device)
            
            # 学习率调度
            total_steps = pretrain_config['epochs'] * len(train_loader)
            lr = get_lr(step_count, total_steps, pretrain_config['learning_rate'])
            for param_group in optimizer.param_groups:
                param_group['lr'] = lr
            
            # 前向传播（混合精度）
            with torch.cuda.amp.autocast(dtype=torch.bfloat16 if device == 'cuda' else torch.float16):
                res = model(input_ids, labels=labels)
                loss = (res.loss + res.aux_loss) / pretrain_config['accumulation_steps']
            
            # 反向传播
            scaler.scale(loss).backward()
            
            # 梯度更新
            if (step + 1) % pretrain_config['accumulation_steps'] == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), pretrain_config['grad_clip'])
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            
            # 记录损失
            current_loss = loss.item() * pretrain_config['accumulation_steps']
            loss_history.append(current_loss)
            epoch_loss += current_loss
            step_count += 1
            
            # 日志输出
            if step % pretrain_config['log_interval'] == 0:
                spend_time = time.time() - start_time
                avg_loss = epoch_loss / (step + 1)
                eta_min = spend_time / max(step, 1) * (len(train_loader) - step) / 60
                Logger(f'Step: {step}/{len(train_loader)}, loss: {current_loss:.4f}, avg_loss: {avg_loss:.4f}, lr: {lr:.8f}, ETA: {eta_min:.1f}min')
            
            # 保存模型
            if step % pretrain_config['save_interval'] == 0 and step > 0:
                moe_suffix = '_moe' if pretrain_config['use_moe'] else ''
                ckp = f"{OUTPUT_DIR}/{pretrain_config['save_weight']}_{pretrain_config['hidden_size']}{moe_suffix}.pth"
                state_dict = {k: v.half().cpu() for k, v in model.state_dict().items()}
                torch.save(state_dict, ckp)
                Logger(f'模型已保存至: {ckp}')
        
        # Epoch 结束
        avg_epoch_loss = epoch_loss / len(train_loader)
        Logger(f"\nEpoch {epoch + 1} 完成! 平均损失: {avg_epoch_loss:.4f}")
    
    # 最终保存
    moe_suffix = '_moe' if pretrain_config['use_moe'] else ''
    ckp = f"{OUTPUT_DIR}/{pretrain_config['save_weight']}_{pretrain_config['hidden_size']}{moe_suffix}.pth"
    state_dict = {k: v.half().cpu() for k, v in model.state_dict().items()}
    torch.save(state_dict, ckp)
    Logger(f"\n预训练完成! 最终模型保存至: {ckp}")

print("预训练函数定义完成！运行 train_pretrain() 开始训练。")

In [ ]:
# 可视化训练损失
def plot_loss():
    plt.figure(figsize=(10, 6))
    plt.plot(loss_history, linewidth=0.5, alpha=0.7)
    # 移动平均
    window = 50
    if len(loss_history) > window:
        smoothed = np.convolve(loss_history, np.ones(window)/window, mode='valid')
        plt.plot(smoothed, linewidth=2, label=f'{window}-step moving average')
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.title('Pretraining Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

print("损失可视化函数定义完成！")

## 第七部分：指令微调 (SFT)

In [ ]:
# SFT 配置（针对 Colab T4 GPU 优化）
sft_config = {
    'epochs': 2,
    'batch_size': 8,  # 更小的 batch size
    'learning_rate': 1e-5,
    'max_seq_len': 256,
    'hidden_size': 768,
    'num_hidden_layers': 8,
    'use_moe': False,
    'accumulation_steps': 4,
    'grad_clip': 1.0,
    'log_interval': 20,
    'save_interval': 200,
    'num_workers': 0,
    'data_path': os.path.join(DATASET_DIR, 'sft_t2t_mini.jsonl'),
    'save_weight': 'full_sft',
    'from_weight': 'pretrain',
}

# 检查数据文件
if not os.path.exists(sft_config['data_path']):
    print(f"⚠️ 数据文件不存在: {sft_config['data_path']}")
else:
    print("✅ SFT 数据文件存在！")

print(f"\nSFT 配置 (Colab 优化版):")
for k, v in sft_config.items():
    print(f"  {k}: {v}")

In [ ]:
# 初始化 SFT 环境
setup_seed(42)

# 创建模型配置
sft_lm_config = MiniMindConfig(
    hidden_size=sft_config['hidden_size'],
    num_hidden_layers=sft_config['num_hidden_layers'],
    use_moe=sft_config['use_moe']
)

# 加载 Tokenizer
sft_tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

# 创建模型并加载预训练权重
sft_model = MiniMindForCausalLM(sft_lm_config)

# 检查预训练权重是否存在
pretrain_weight_path = f"{OUTPUT_DIR}/{sft_config['from_weight']}_{sft_config['hidden_size']}.pth"
if os.path.exists(pretrain_weight_path):
    weights = torch.load(pretrain_weight_path, map_location=device)
    sft_model.load_state_dict(weights, strict=False)
    Logger(f"已加载预训练权重: {pretrain_weight_path}")
else:
    Logger("⚠️ 未找到预训练权重，将从随机初始化开始")

get_model_params(sft_model, sft_lm_config)
sft_model = sft_model.to(device)

# 创建 SFT 数据集和数据加载器
sft_ds = SFTDataset(sft_config['data_path'], sft_tokenizer, max_length=sft_config['max_seq_len'])
sft_loader = DataLoader(sft_ds, batch_size=sft_config['batch_size'], shuffle=True, num_workers=sft_config['num_workers'])

# 优化器和混合精度
sft_optimizer = optim.AdamW(sft_model.parameters(), lr=sft_config['learning_rate'])
sft_scaler = torch.cuda.amp.GradScaler(enabled=True)

print("\nSFT 环境初始化完成！")
print(f"训练样本数: {len(sft_ds)}")
print(f"每个 epoch 步数: {len(sft_loader)}")

In [ ]:
# SFT 训练损失记录
sft_loss_history = []
sft_step_count = 0

# SFT 训练循环
def train_sft():
    global sft_step_count, sft_loss_history
    sft_model.train()
    
    for epoch in range(sft_config['epochs']):
        Logger(f"\n{'='*50}")
        Logger(f"SFT Epoch [{epoch + 1}/{sft_config['epochs']}]")
        Logger(f"{'='*50}")
        
        epoch_loss = 0
        start_time = time.time()
        
        for step, (input_ids, labels) in enumerate(sft_loader):
            input_ids = input_ids.to(device)
            labels = labels.to(device)
            
            # 学习率调度
            total_steps = sft_config['epochs'] * len(sft_loader)
            lr = get_lr(sft_step_count, total_steps, sft_config['learning_rate'])
            for param_group in sft_optimizer.param_groups:
                param_group['lr'] = lr
            
            # 前向传播（混合精度）
            with torch.cuda.amp.autocast(dtype=torch.bfloat16 if device == 'cuda' else torch.float16):
                res = sft_model(input_ids, labels=labels)
                loss = (res.loss + res.aux_loss) / sft_config['accumulation_steps']
            
            # 反向传播
            sft_scaler.scale(loss).backward()
            
            # 梯度更新
            if (step + 1) % sft_config['accumulation_steps'] == 0:
                sft_scaler.unscale_(sft_optimizer)
                torch.nn.utils.clip_grad_norm_(sft_model.parameters(), sft_config['grad_clip'])
                sft_scaler.step(sft_optimizer)
                sft_scaler.update()
                sft_optimizer.zero_grad(set_to_none=True)
            
            # 记录损失
            current_loss = loss.item() * sft_config['accumulation_steps']
            sft_loss_history.append(current_loss)
            epoch_loss += current_loss
            sft_step_count += 1
            
            # 日志输出
            if step % sft_config['log_interval'] == 0:
                spend_time = time.time() - start_time
                avg_loss = epoch_loss / (step + 1)
                eta_min = spend_time / max(step, 1) * (len(sft_loader) - step) / 60
                Logger(f'Step: {step}/{len(sft_loader)}, loss: {current_loss:.4f}, avg_loss: {avg_loss:.4f}, lr: {lr:.8f}, ETA: {eta_min:.1f}min')
            
            # 保存模型
            if step % sft_config['save_interval'] == 0 and step > 0:
                moe_suffix = '_moe' if sft_config['use_moe'] else ''
                ckp = f"{OUTPUT_DIR}/{sft_config['save_weight']}_{sft_config['hidden_size']}{moe_suffix}.pth"
                state_dict = {k: v.half().cpu() for k, v in sft_model.state_dict().items()}
                torch.save(state_dict, ckp)
                Logger(f'模型已保存至: {ckp}')
        
        # Epoch 结束
        avg_epoch_loss = epoch_loss / len(sft_loader)
        Logger(f"\nSFT Epoch {epoch + 1} 完成! 平均损失: {avg_epoch_loss:.4f}")
    
    # 最终保存
    moe_suffix = '_moe' if sft_config['use_moe'] else ''
    ckp = f"{OUTPUT_DIR}/{sft_config['save_weight']}_{sft_config['hidden_size']}{moe_suffix}.pth"
    state_dict = {k: v.half().cpu() for k, v in sft_model.state_dict().items()}
    torch.save(state_dict, ckp)
    Logger(f"\nSFT 完成! 最终模型保存至: {ckp}")

print("SFT 训练函数定义完成！运行 train_sft() 开始训练。")

In [ ]:
# 可视化 SFT 训练损失
def plot_sft_loss():
    plt.figure(figsize=(10, 6))
    plt.plot(sft_loss_history, linewidth=0.5, alpha=0.7)
    window = 50
    if len(sft_loss_history) > window:
        smoothed = np.convolve(sft_loss_history, np.ones(window)/window, mode='valid')
        plt.plot(smoothed, linewidth=2, label=f'{window}-step moving average')
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.title('SFT Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

print("SFT 损失可视化函数定义完成！")

## 第八部分：GRPO 强化学习训练（可选）

In [ ]:
# GRPO (Group Relative Policy Optimization) 是一种强化学习方法
# 它通过生成多个响应并根据奖励进行优化

# GRPO 配置
grpo_config = {
    'epochs': 1,
    'batch_size': 2,
    'learning_rate': 3e-7,
    'max_seq_len': 512,
    'max_gen_len': 512,
    'num_generations': 4,  # 每个 prompt 生成的样本数
    'beta': 0.1,  # KL 惩罚系数
    'data_path': os.path.join(DATASET_DIR, 'rlaif.jsonl'),
}

print("GRPO 配置:")
for k, v in grpo_config.items():
    print(f"  {k}: {v}")

In [ ]:
# GRPO 奖励计算函数
import re

def rep_penalty(text, n=3, cap=0.5):
    """重复惩罚"""
    toks = re.findall(r"\w+|[^\w\s]", text.lower())
    grams = [tuple(toks[i:i + n]) for i in range(len(toks) - n + 1)]
    return min(cap, (len(grams) - len(set(grams))) * cap * 2 / len(grams)) if grams else 0.0

def calculate_rewards(responses):
    """计算奖励分数"""
    rewards = []
    for response in responses:
        reward = 0.0
        # 长度奖励
        if 20 <= len(response.strip()) <= 800:
            reward += 0.5
        else:
            reward -= 0.5
        
        # 思考标签奖励
        if '</think>' in response:
            thinking_content, answer_content = response.split('</think>', 1)
            if 20 <= len(thinking_content.strip()) <= 300:
                reward += 1.0
            else:
                reward -= 0.5
            if response.count('</think>') == 1:
                reward += 0.25
            else:
                reward -= 0.25
            answer = answer_content.strip()
        else:
            answer = response.strip()
        
        # 重复惩罚
        reward -= rep_penalty(answer)
        
        rewards.append(reward)
    return torch.tensor(rewards, device=device)

print("GRPO 奖励函数定义完成！")

In [ ]:
# GRPO 训练循环
grpo_loss_history = []
grpo_step_count = 0

def train_grpo():
    global grpo_step_count, grpo_loss_history
    dpo_model.train()  # 使用 DPO 后的模型
    
    # 创建参考模型（用于 KL 散度计算）
    ref_model_grpo = MiniMindForCausalLM(inf_config)
    if os.path.exists(f"{OUTPUT_DIR}/dpo_{inf_config.hidden_size}.pth"):
        weights = torch.load(f"{OUTPUT_DIR}/dpo_{inf_config.hidden_size}.pth", map_location=device)
        ref_model_grpo.load_state_dict(weights, strict=False)
    elif os.path.exists(weight_path):
        weights = torch.load(weight_path, map_location=device)
        ref_model_grpo.load_state_dict(weights, strict=False)
    ref_model_grpo = ref_model_grpo.eval().to(device)
    for param in ref_model_grpo.parameters():
        param.requires_grad = False
    
    # 加载 RLAIF 数据
    if not os.path.exists(grpo_config['data_path']):
        print(f"⚠️ GRPO 数据文件不存在: {grpo_config['data_path']}")
        print("跳过 GRPO 训练")
        return
    
    rlaif_ds = load_dataset('json', data_files=grpo_config['data_path'], split='train')
    print(f"RLAIF 数据集加载完成: {len(rlaif_ds)} 样本")
    
    grpo_optimizer = optim.AdamW(dpo_model.parameters(), lr=grpo_config['learning_rate'])
    
    for epoch in range(grpo_config['epochs']):
        Logger(f"\n{'='*50}")
        Logger(f"GRPO Epoch [{epoch + 1}/{grpo_config['epochs']}]")
        Logger(f"{'='*50}")
        
        start_time = time.time()
        
        for step, sample in enumerate(rlaif_ds):
            prompt = sample['conversations'][:-1]  # 去掉最后一个 assistant 回复
            prompt_text = inf_tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
            
            # 编码 prompt
            prompt_inputs = inf_tokenizer(prompt_text, return_tensors="pt", padding=True, 
                                          return_token_type_ids=False, padding_side="left",
                                          add_special_tokens=False).to(device)
            
            # 生成多个响应
            all_responses = []
            all_completion_ids = []
            for _ in range(grpo_config['num_generations']):
                with torch.no_grad():
                    generated = dpo_model.generate(
                        inputs=prompt_inputs["input_ids"],
                        attention_mask=prompt_inputs["attention_mask"],
                        max_new_tokens=grpo_config['max_gen_len'],
                        do_sample=True,
                        temperature=0.8,
                        top_p=0.95,
                        pad_token_id=inf_tokenizer.pad_token_id,
                        eos_token_id=inf_tokenizer.eos_token_id,
                    )
                completion = generated[0][prompt_inputs["input_ids"].shape[1]:]
                all_completion_ids.append(completion)
                response_text = inf_tokenizer.decode(completion, skip_special_tokens=True)
                all_responses.append(response_text)
            
            # 计算奖励
            rewards = calculate_rewards(all_responses)
            
            # 计算优势函数
            mean_r = rewards.mean()
            std_r = rewards.std() + 1e-4
            advantages = (rewards - mean_r) / std_r
            
            # 计算策略梯度
            total_loss = 0
            for i, (completion, adv) in enumerate(zip(all_completion_ids, advantages)):
                # 计算当前策略下的 log probs
                outputs = dpo_model(completion.unsqueeze(0))
                logits = outputs.logits[:, :-1, :]
                log_probs = F.log_softmax(logits, dim=-1).gather(2, completion[1:].unsqueeze(-1)).squeeze(-1)
                
                # 计算参考策略下的 log probs
                with torch.no_grad():
                    ref_outputs = ref_model_grpo(completion.unsqueeze(0))
                    ref_logits = ref_outputs.logits[:, :-1, :]
                    ref_log_probs = F.log_softmax(ref_logits, dim=-1).gather(2, completion[1:].unsqueeze(-1)).squeeze(-1)
                
                # KL 散度
                kl_div = ref_log_probs - log_probs
                per_token_kl = torch.exp(kl_div) - kl_div - 1
                
                # 策略损失
                policy_loss = -(log_probs.mean() * adv - grpo_config['beta'] * per_token_kl.mean())
                total_loss += policy_loss
            
            total_loss = total_loss / grpo_config['num_generations']
            
            # 反向传播
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(dpo_model.parameters(), 1.0)
            grpo_optimizer.step()
            grpo_optimizer.zero_grad()
            
            grpo_loss_history.append(total_loss.item())
            grpo_step_count += 1
            
            if step % 10 == 0:
                spend_time = time.time() - start_time
                eta_min = spend_time / max(step, 1) * (len(rlaif_ds) - step) / 60
                Logger(f'Step: {step}/{len(rlaif_ds)}, loss: {total_loss.item():.4f}, '
                       f'reward_mean: {rewards.mean().item():.4f}, reward_std: {rewards.std().item():.4f}, '
                       f'ETA: {eta_min:.1f}min')
        
        Logger(f"\nGRPO Epoch {epoch + 1} 完成!")
    
    # 保存模型
    ckp = f"{OUTPUT_DIR}/grpo_{inf_config.hidden_size}.pth"
    state_dict = {k: v.half().cpu() for k, v in dpo_model.state_dict().items()}
    torch.save(state_dict, ckp)
    Logger(f"\nGRPO 完成! 模型保存至: {ckp}")

print("GRPO 训练函数定义完成！运行 train_grpo() 开始训练。")

## 第九部分：PPO 强化学习训练（可选）

In [ ]:
# PPO (Proximal Policy Optimization) 是一种经典强化学习方法
# 它使用 Actor-Critic 架构，通过奖励模型进行优化

# Critic 模型定义
class CriticModel(MiniMindForCausalLM):
    def __init__(self, params):
        super().__init__(params)
        self.value_head = nn.Linear(params.hidden_size, 1)

    def forward(self, input_ids=None, attention_mask=None, **kwargs):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        hidden_states = self.model.norm(outputs[0])
        values = self.value_head(hidden_states).squeeze(-1)
        return values

print("CriticModel 定义完成！")

In [ ]:
# PPO 配置
ppo_config = {
    'epochs': 1,
    'batch_size': 2,
    'learning_rate': 1e-6,
    'max_seq_len': 512,
    'max_gen_len': 512,
    'ppo_epochs': 3,  # PPO 内部迭代次数
    'clip_epsilon': 0.2,  # PPO clip 参数
    'beta': 0.1,  # KL 惩罚系数
    'data_path': os.path.join(DATASET_DIR, 'rlaif.jsonl'),
}

print("PPO 配置:")
for k, v in ppo_config.items():
    print(f"  {k}: {v}")

In [ ]:
# PPO 训练循环
ppo_loss_history = []
ppo_step_count = 0

def train_ppo():
    global ppo_step_count, ppo_loss_history
    
    # 检查数据文件
    if not os.path.exists(ppo_config['data_path']):
        print(f"⚠️ PPO 数据文件不存在: {ppo_config['data_path']}")
        print("跳过 PPO 训练")
        return
    
    # 加载 Actor 模型
    actor_model = MiniMindForCausalLM(inf_config)
    if os.path.exists(f"{OUTPUT_DIR}/grpo_{inf_config.hidden_size}.pth"):
        weights = torch.load(f"{OUTPUT_DIR}/grpo_{inf_config.hidden_size}.pth", map_location=device)
        actor_model.load_state_dict(weights, strict=False)
    elif os.path.exists(f"{OUTPUT_DIR}/dpo_{inf_config.hidden_size}.pth"):
        weights = torch.load(f"{OUTPUT_DIR}/dpo_{inf_config.hidden_size}.pth", map_location=device)
        actor_model.load_state_dict(weights, strict=False)
    elif os.path.exists(weight_path):
        weights = torch.load(weight_path, map_location=device)
        actor_model.load_state_dict(weights, strict=False)
    actor_model = actor_model.to(device)
    
    # 创建 Critic 模型
    critic_model = CriticModel(inf_config)
    critic_model = critic_model.to(device)
    
    # 创建参考模型
    ref_model = MiniMindForCausalLM(inf_config)
    ref_model.load_state_dict(actor_model.state_dict(), strict=False)
    ref_model = ref_model.eval().to(device)
    for param in ref_model.parameters():
        param.requires_grad = False
    
    # 优化器
    actor_optimizer = optim.AdamW(actor_model.parameters(), lr=ppo_config['learning_rate'])
    critic_optimizer = optim.AdamW(critic_model.parameters(), lr=ppo_config['learning_rate'])
    
    # 加载数据
    rlaif_ds = load_dataset('json', data_files=ppo_config['data_path'], split='train')
    print(f"RLAIF 数据集加载完成: {len(rlaif_ds)} 样本")
    
    for epoch in range(ppo_config['epochs']):
        Logger(f"\n{'='*50}")
        Logger(f"PPO Epoch [{epoch + 1}/{ppo_config['epochs']}]")
        Logger(f"{'='*50}")
        
        start_time = time.time()
        
        for step, sample in enumerate(rlaif_ds):
            prompt = sample['conversations'][:-1]
            prompt_text = inf_tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
            
            # 编码
            prompt_inputs = inf_tokenizer(prompt_text, return_tensors="pt", padding=True,
                                          padding_side="left", add_special_tokens=False).to(device)
            prompt_length = prompt_inputs["input_ids"].shape[1]
            
            # 生成响应
            with torch.no_grad():
                generated = actor_model.generate(
                    inputs=prompt_inputs["input_ids"],
                    attention_mask=prompt_inputs["attention_mask"],
                    max_new_tokens=ppo_config['max_gen_len'],
                    do_sample=True,
                    temperature=0.8,
                    top_p=0.95,
                    pad_token_id=inf_tokenizer.pad_token_id,
                    eos_token_id=inf_tokenizer.eos_token_id,
                )
            
            response_ids = generated[0][prompt_length:]
            response_text = inf_tokenizer.decode(response_ids, skip_special_tokens=True)
            
            # 计算奖励（简化版）
            reward = 0.0
            if 20 <= len(response_text.strip()) <= 800:
                reward += 0.5
            if '</think>' in response_text:
                thinking, answer = response_text.split('</think>', 1)
                if 20 <= len(thinking.strip()) <= 300:
                    reward += 1.0
            
            # PPO 内部迭代
            for ppo_epoch in range(ppo_config['ppo_epochs']):
                # 计算 old log probs
                with torch.no_grad():
                    outputs = actor_model(response_ids.unsqueeze(0))
                    logits = outputs.logits[:, :-1, :]
                    old_log_probs = F.log_softmax(logits, dim=-1).gather(2, response_ids[1:].unsqueeze(-1)).squeeze(-1)
                    
                    # 价值估计
                    values = critic_model(response_ids.unsqueeze(0))
                
                # 计算优势
                advantages = torch.tensor([reward], device=device)
                returns = advantages + values.mean()
                
                # Actor 更新
                new_outputs = actor_model(response_ids.unsqueeze(0))
                new_logits = new_outputs.logits[:, :-1, :]
                new_log_probs = F.log_softmax(new_logits, dim=-1).gather(2, response_ids[1:].unsqueeze(-1)).squeeze(-1)
                
                ratio = torch.exp(new_log_probs - old_log_probs)
                clipped_ratio = torch.clamp(ratio, 1 - ppo_config['clip_epsilon'], 1 + ppo_config['clip_epsilon'])
                
                actor_loss = -torch.min(ratio * advantages, clipped_ratio * advantages).mean()
                
                actor_optimizer.zero_grad()
                actor_loss.backward()
                torch.nn.utils.clip_grad_norm_(actor_model.parameters(), 1.0)
                actor_optimizer.step()
                
                # Critic 更新
                new_values = critic_model(response_ids.unsqueeze(0))
                critic_loss = F.mse_loss(new_values.mean(), returns)
                
                critic_optimizer.zero_grad()
                critic_loss.backward()
                torch.nn.utils.clip_grad_norm_(critic_model.parameters(), 1.0)
                critic_optimizer.step()
            
            ppo_loss_history.append(actor_loss.item())
            ppo_step_count += 1
            
            if step % 10 == 0:
                spend_time = time.time() - start_time
                eta_min = spend_time / max(step, 1) * (len(rlaif_ds) - step) / 60
                Logger(f'Step: {step}/{len(rlaif_ds)}, actor_loss: {actor_loss.item():.4f}, '
                       f'critic_loss: {critic_loss.item():.4f}, reward: {reward:.4f}, '
                       f'ETA: {eta_min:.1f}min')
        
        Logger(f"\nPPO Epoch {epoch + 1} 完成!")
    
    # 保存模型
    ckp = f"{OUTPUT_DIR}/ppo_{inf_config.hidden_size}.pth"
    state_dict = {k: v.half().cpu() for k, v in actor_model.state_dict().items()}
    torch.save(state_dict, ckp)
    Logger(f"\nPPO 完成! 模型保存至: {ckp}")

print("PPO 训练函数定义完成！运行 train_ppo() 开始训练。")

## 第十部分：LoRA 低秩微调（可选）

In [ ]:
# LoRA (Low-Rank Adaptation) 是一种高效的微调方法
# 它只在模型的线性层中添加低秩矩阵，大幅减少可训练参数

# 定义 LoRA 模块
class LoRA(nn.Module):
    def __init__(self, in_features, out_features, rank):
        super().__init__()
        self.rank = rank
        self.A = nn.Linear(in_features, rank, bias=False)
        self.B = nn.Linear(rank, out_features, bias=False)
        self.A.weight.data.normal_(mean=0.0, std=0.02)
        self.B.weight.data.zero_()

    def forward(self, x):
        return self.B(self.A(x))

def apply_lora(model, rank=16):
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and module.weight.shape[0] == module.weight.shape[1]:
            lora = LoRA(module.weight.shape[0], module.weight.shape[1], rank=rank).to(model.device)
            setattr(module, "lora", lora)
            original_forward = module.forward
            def forward_with_lora(x, layer1=original_forward, layer2=lora):
                return layer1(x) + layer2(x)
            module.forward = forward_with_lora

def save_lora(model, path):
    state_dict = {}
    for name, module in model.named_modules():
        if hasattr(module, 'lora'):
            lora_state = {f'{name}.lora.{k}': v.cpu().half() for k, v in module.lora.state_dict().items()}
            state_dict.update(lora_state)
    torch.save(state_dict, path)

def load_lora(model, path):
    state_dict = torch.load(path, map_location=model.device)
    for name, module in model.named_modules():
        if hasattr(module, 'lora'):
            lora_state = {k.replace(f'{name}.lora.', ''): v for k, v in state_dict.items() if f'{name}.lora.' in k}
            module.lora.load_state_dict(lora_state)

def merge_lora(model, lora_path, save_path):
    load_lora(model, lora_path)
    state_dict = {k: v.cpu().half() for k, v in model.state_dict().items() if '.lora.' not in k}
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and '.lora.' not in name:
            state_dict[f'{name}.weight'] = module.weight.data.clone().cpu().half()
            if hasattr(module, 'lora'):
                state_dict[f'{name}.weight'] += (module.lora.B.weight.data @ module.lora.A.weight.data).cpu().half()
    torch.save(state_dict, save_path)

print("LoRA 模块定义完成！")

In [ ]:
# LoRA 配置
lora_config = {
    'rank': 16,  # LoRA 秩
    'epochs': 5,
    'batch_size': 8,
    'learning_rate': 1e-4,
    'max_seq_len': 256,
    'accumulation_steps': 4,
    'log_interval': 20,
    'lora_name': 'lora_custom',
}

print("LoRA 配置:")
for k, v in lora_config.items():
    print(f"  {k}: {v}")

In [ ]:
# 初始化 LoRA 微调环境
setup_seed(42)

# 加载基础模型（使用 SFT 后的权重）
lora_model = MiniMindForCausalLM(inf_config)
if os.path.exists(weight_path):
    weights = torch.load(weight_path, map_location=device)
    lora_model.load_state_dict(weights, strict=False)
    print(f"已加载基础权重: {weight_path}")

# 应用 LoRA
apply_lora(lora_model, rank=lora_config['rank'])
lora_model = lora_model.to(device)

# 冻结所有参数，只训练 LoRA
for name, param in lora_model.named_parameters():
    if '.lora.' not in name:
        param.requires_grad = False

# 收集可训练参数
lora_params = [p for p in lora_model.parameters() if p.requires_grad]
print(f"\n可训练参数数量: {sum(p.numel() for p in lora_params) / 1e6:.3f}M")
print(f"总参数数量: {sum(p.numel() for p in lora_model.parameters()) / 1e6:.3f}M")

# 优化器（只优化 LoRA 参数）
lora_optimizer = optim.AdamW(lora_params, lr=lora_config['learning_rate'])
lora_scaler = torch.cuda.amp.GradScaler(enabled=True)

print("\nLoRA 微调环境初始化完成！")

In [ ]:
# LoRA 训练循环
lora_loss_history = []
lora_step_count = 0

def train_lora():
    global lora_step_count, lora_loss_history
    lora_model.train()
    
    for epoch in range(lora_config['epochs']):
        Logger(f"\n{'='*50}")
        Logger(f"LoRA Epoch [{epoch + 1}/{lora_config['epochs']}]")
        Logger(f"{'='*50}")
        
        epoch_loss = 0
        start_time = time.time()
        
        for step, (input_ids, labels) in enumerate(sft_loader):  # 复用 SFT 数据加载器
            input_ids = input_ids.to(device)
            labels = labels.to(device)
            
            lr = get_lr(lora_step_count, lora_config['epochs'] * len(sft_loader), lora_config['learning_rate'])
            for param_group in lora_optimizer.param_groups:
                param_group['lr'] = lr
            
            with torch.cuda.amp.autocast(dtype=torch.bfloat16 if device == 'cuda' else torch.float16):
                res = lora_model(input_ids, labels=labels)
                loss = (res.loss + res.aux_loss) / lora_config['accumulation_steps']
            
            lora_scaler.scale(loss).backward()
            
            if (step + 1) % lora_config['accumulation_steps'] == 0:
                lora_scaler.unscale_(lora_optimizer)
                torch.nn.utils.clip_grad_norm_(lora_params, lora_config['grad_clip'] if hasattr(lora_config, 'grad_clip') else 1.0)
                lora_scaler.step(lora_optimizer)
                lora_scaler.update()
                lora_optimizer.zero_grad(set_to_none=True)
            
            current_loss = loss.item() * lora_config['accumulation_steps']
            lora_loss_history.append(current_loss)
            epoch_loss += current_loss
            lora_step_count += 1
            
            if step % lora_config['log_interval'] == 0:
                spend_time = time.time() - start_time
                avg_loss = epoch_loss / (step + 1)
                eta_min = spend_time / max(step, 1) * (len(sft_loader) - step) / 60
                Logger(f'Step: {step}/{len(sft_loader)}, loss: {current_loss:.4f}, avg_loss: {avg_loss:.4f}, lr: {lr:.8f}, ETA: {eta_min:.1f}min')
        
        avg_epoch_loss = epoch_loss / len(sft_loader)
        Logger(f"\nLoRA Epoch {epoch + 1} 完成! 平均损失: {avg_epoch_loss:.4f}")
    
    # 保存 LoRA 权重
    lora_path = f"{OUTPUT_DIR}/{lora_config['lora_name']}_{lora_config['rank']}.pth"
    save_lora(lora_model, lora_path)
    Logger(f"\nLoRA 微调完成! 权重保存至: {lora_path}")

print("LoRA 训练函数定义完成！运行 train_lora() 开始训练。")

## 第十一部分：模型蒸馏（可选）

In [ ]:
# 模型蒸馏：使用更大的教师模型指导学生模型
# 通过 KL 散度损失让学生模仿教师的输出分布

# 蒸馏损失函数
def distillation_loss(student_logits, teacher_logits, temperature=1.0, reduction='batchmean'):
    with torch.no_grad():
        teacher_probs = F.softmax(teacher_logits / temperature, dim=-1).detach()
    student_log_probs = F.log_softmax(student_logits / temperature, dim=-1)
    kl = F.kl_div(student_log_probs, teacher_probs, reduction=reduction)
    return (temperature ** 2) * kl

print("蒸馏损失函数定义完成！")

In [ ]:
# 蒸馏配置
distill_config = {
    'epochs': 3,
    'batch_size': 8,
    'learning_rate': 1e-5,
    'max_seq_len': 256,
    'accumulation_steps': 4,
    'alpha': 0.3,  # CE 损失权重
    'temperature': 2.0,  # 蒸馏温度
    'teacher_hidden_size': 1024,  # 教师模型维度
    'student_hidden_size': 768,   # 学生模型维度
}

print("蒸馏配置:")
for k, v in distill_config.items():
    print(f"  {k}: {v}")

In [ ]:
# 初始化蒸馏环境
setup_seed(42)

# 学生模型配置
student_config = MiniMindConfig(
    hidden_size=distill_config['student_hidden_size'],
    num_hidden_layers=8,
    use_moe=False
)

# 教师模型配置（更大）
teacher_config = MiniMindConfig(
    hidden_size=distill_config['teacher_hidden_size'],
    num_hidden_layers=12,
    use_moe=False
)

# 创建学生模型
student_model = MiniMindForCausalLM(student_config).to(device)

# 创建教师模型（如果有预训练权重则加载）
teacher_model = MiniMindForCausalLM(teacher_config).to(device)
teacher_model.eval()
for param in teacher_model.parameters():
    param.requires_grad = False

# 优化器
distill_optimizer = optim.AdamW(student_model.parameters(), lr=distill_config['learning_rate'])
distill_scaler = torch.cuda.amp.GradScaler(enabled=True)

print("蒸馏环境初始化完成！")

In [ ]:
# 蒸馏训练循环
distill_loss_history = []
distill_step_count = 0

def train_distill():
    global distill_step_count, distill_loss_history
    student_model.train()
    
    for epoch in range(distill_config['epochs']):
        Logger(f"\n{'='*50}")
        Logger(f"Distill Epoch [{epoch + 1}/{distill_config['epochs']}]")
        Logger(f"{'='*50}")
        
        epoch_loss = 0
        start_time = time.time()
        
        for step, (input_ids, labels) in enumerate(sft_loader):  # 复用 SFT 数据
            input_ids = input_ids.to(device)
            labels = labels.to(device)
            
            lr = get_lr(distill_step_count, distill_config['epochs'] * len(sft_loader), distill_config['learning_rate'])
            for param_group in distill_optimizer.param_groups:
                param_group['lr'] = lr
            
            # 学生模型前向传播
            with torch.cuda.amp.autocast(dtype=torch.bfloat16 if device == 'cuda' else torch.float16):
                student_outputs = student_model(input_ids, labels=labels)
                student_logits = student_outputs.logits[..., :-1, :].contiguous()
            
            # 教师模型前向传播（no_grad）
            with torch.no_grad():
                teacher_outputs = teacher_model(input_ids)
                teacher_logits = teacher_outputs.logits[..., :-1, :].contiguous()
                # 对齐词表大小
                vocab_size_student = student_logits.size(-1)
                teacher_logits = teacher_logits[..., :vocab_size_student]
            
            # 计算 CE 损失
            ce_loss = F.cross_entropy(
                student_logits.view(-1, student_logits.size(-1)),
                labels[..., 1:].contiguous().view(-1),
                ignore_index=-100
            )
            
            # 计算蒸馏损失
            loss_mask = (labels[..., 1:] != -100).float().view(-1)
            d_loss = distillation_loss(
                student_logits.view(-1, student_logits.size(-1))[loss_mask == 1],
                teacher_logits.view(-1, teacher_logits.size(-1))[loss_mask == 1],
                temperature=distill_config['temperature']
            )
            
            # 总损失
            alpha = distill_config['alpha']
            loss = (alpha * ce_loss + (1 - alpha) * d_loss) / distill_config['accumulation_steps']
            
            distill_scaler.scale(loss).backward()
            
            if (step + 1) % distill_config['accumulation_steps'] == 0:
                distill_scaler.unscale_(distill_optimizer)
                torch.nn.utils.clip_grad_norm_(student_model.parameters(), 1.0)
                distill_scaler.step(distill_optimizer)
                distill_scaler.update()
                distill_optimizer.zero_grad(set_to_none=True)
            
            current_loss = loss.item() * distill_config['accumulation_steps']
            distill_loss_history.append(current_loss)
            epoch_loss += current_loss
            distill_step_count += 1
            
            if step % 20 == 0:
                spend_time = time.time() - start_time
                avg_loss = epoch_loss / (step + 1)
                eta_min = spend_time / max(step, 1) * (len(sft_loader) - step) / 60
                Logger(f'Step: {step}/{len(sft_loader)}, loss: {current_loss:.4f}, ce: {ce_loss.item():.4f}, '
                       f'distill: {d_loss.item():.4f}, lr: {lr:.8f}, ETA: {eta_min:.1f}min')
        
        avg_epoch_loss = epoch_loss / len(sft_loader)
        Logger(f"\nDistill Epoch {epoch + 1} 完成! 平均损失: {avg_epoch_loss:.4f}")
    
    # 保存学生模型
    ckp = f"{OUTPUT_DIR}/distill_{student_config.hidden_size}.pth"
    state_dict = {k: v.half().cpu() for k, v in student_model.state_dict().items()}
    torch.save(state_dict, ckp)
    Logger(f"\n蒸馏完成! 学生模型保存至: {ckp}")

print("蒸馏训练函数定义完成！运行 train_distill() 开始训练。")

## 第十二部分：DPO 偏好对齐训练（可选）

In [ ]:
# DPO (Direct Preference Optimization) 是一种直接偏好优化方法
# 它使用偏好数据（chosen vs rejected）来对齐模型输出

# DPO 数据集类
class DPODataset(Dataset):
    def __init__(self, file_path, tokenizer, max_length=4096):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.padding = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
        self.bos_id = tokenizer(f'{tokenizer.bos_token}assistant\n', add_special_tokens=False).input_ids
        self.eos_id = tokenizer(f'{tokenizer.eos_token}\n', add_special_tokens=False).input_ids
        self.samples = load_dataset('json', data_files=file_path, split='train')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        chosen = sample['chosen']
        rejected = sample['rejected']
        chosen_prompt = self.tokenizer.apply_chat_template(
            chosen, tokenize=False, add_generation_prompt=False
        )
        rejected_prompt = self.tokenizer.apply_chat_template(
            rejected, tokenize=False, add_generation_prompt=False
        )
        chosen_encoding = self.tokenizer(
            chosen_prompt, truncation=True, max_length=self.max_length, padding='max_length'
        )
        rejected_encoding = self.tokenizer(
            rejected_prompt, truncation=True, max_length=self.max_length, padding='max_length'
        )
        chosen_input_ids = chosen_encoding['input_ids']
        chosen_loss_mask = self.generate_loss_mask(chosen_input_ids)
        rejected_input_ids = rejected_encoding['input_ids']
        rejected_loss_mask = self.generate_loss_mask(rejected_input_ids)
        x_chosen = torch.tensor(chosen_input_ids[:-1], dtype=torch.long)
        y_chosen = torch.tensor(chosen_input_ids[1:], dtype=torch.long)
        mask_chosen = torch.tensor(chosen_loss_mask[1:], dtype=torch.long)
        x_rejected = torch.tensor(rejected_input_ids[:-1], dtype=torch.long)
        y_rejected = torch.tensor(rejected_input_ids[1:], dtype=torch.long)
        mask_rejected = torch.tensor(rejected_loss_mask[1:], dtype=torch.long)
        return {
            'x_chosen': x_chosen,
            'y_chosen': y_chosen,
            'mask_chosen': mask_chosen,
            'x_rejected': x_rejected,
            'y_rejected': y_rejected,
            'mask_rejected': mask_rejected
        }

    def generate_loss_mask(self, input_ids):
        loss_mask = [0] * len(input_ids)
        i = 0
        while i < len(input_ids):
            if input_ids[i:i + len(self.bos_id)] == self.bos_id:
                start = i + len(self.bos_id)
                end = start
                while end < len(input_ids):
                    if input_ids[end:end + len(self.eos_id)] == self.eos_id:
                        break
                    end += 1
                for j in range(start, min(end + len(self.eos_id), self.max_length)):
                    loss_mask[j] = 1
                i = end + len(self.eos_id) if end < len(input_ids) else len(input_ids)
            else:
                i += 1
        return loss_mask

print("DPODataset 定义完成！")

In [ ]:
# DPO 损失函数
def logits_to_log_probs(logits, labels):
    log_probs = F.log_softmax(logits, dim=2)
    log_probs_per_token = torch.gather(log_probs, dim=2, index=labels.unsqueeze(2)).squeeze(-1)
    return log_probs_per_token

def dpo_loss(ref_log_probs, policy_log_probs, mask, beta):
    ref_log_probs = (ref_log_probs * mask).sum(dim=1)
    policy_log_probs = (policy_log_probs * mask).sum(dim=1)
    batch_size = ref_log_probs.shape[0]
    chosen_ref_log_probs = ref_log_probs[:batch_size // 2]
    reject_ref_log_probs = ref_log_probs[batch_size // 2:]
    chosen_policy_log_probs = policy_log_probs[:batch_size // 2]
    reject_policy_log_probs = policy_log_probs[batch_size // 2:]
    pi_logratios = chosen_policy_log_probs - reject_policy_log_probs
    ref_logratios = chosen_ref_log_probs - reject_ref_log_probs
    logits = pi_logratios - ref_logratios
    loss = -F.logsigmoid(beta * logits)
    return loss.mean()

print("DPO 损失函数定义完成！")

In [ ]:
# DPO 配置
dpo_config = {
    'epochs': 2,
    'batch_size': 4,
    'learning_rate': 5e-6,
    'max_seq_len': 512,
    'beta': 0.1,  # DPO 温度参数
    'accumulation_steps': 4,
    'data_path': os.path.join(DATASET_DIR, 'dpo.jsonl'),
}

print("DPO 配置:")
for k, v in dpo_config.items():
    print(f"  {k}: {v}")

In [ ]:
# 初始化 DPO 训练环境
setup_seed(42)

# 加载 SFT 模型作为策略模型
dpo_model = MiniMindForCausalLM(inf_config)
if os.path.exists(weight_path):
    weights = torch.load(weight_path, map_location=device)
    dpo_model.load_state_dict(weights, strict=False)
    print(f"已加载 SFT 权重: {weight_path}")

# 创建参考模型（冻结）
ref_model = MiniMindForCausalLM(inf_config)
if os.path.exists(weight_path):
    weights = torch.load(weight_path, map_location=device)
    ref_model.load_state_dict(weights, strict=False)
ref_model = ref_model.eval().to(device)
for param in ref_model.parameters():
    param.requires_grad = False

dpo_model = dpo_model.to(device)

# 创建 DPO 数据集和加载器
if os.path.exists(dpo_config['data_path']):
    dpo_ds = DPODataset(dpo_config['data_path'], inf_tokenizer, max_length=dpo_config['max_seq_len'])
    dpo_loader = DataLoader(dpo_ds, batch_size=dpo_config['batch_size'], shuffle=True, num_workers=0)
    print(f"DPO 数据集加载完成: {len(dpo_ds)} 样本")
else:
    print(f"⚠️ DPO 数据文件不存在: {dpo_config['data_path']}")
    print("跳过 DPO 初始化")

# 优化器
dpo_optimizer = optim.AdamW(dpo_model.parameters(), lr=dpo_config['learning_rate'])
dpo_scaler = torch.cuda.amp.GradScaler(enabled=True)

print("DPO 环境初始化完成！")

In [ ]:
# DPO 训练循环
dpo_loss_history = []
dpo_step_count = 0

def train_dpo():
    global dpo_step_count, dpo_loss_history
    dpo_model.train()
    
    for epoch in range(dpo_config['epochs']):
        Logger(f"\n{'='*50}")
        Logger(f"DPO Epoch [{epoch + 1}/{dpo_config['epochs']}]")
        Logger(f"{'='*50}")
        
        epoch_loss = 0
        start_time = time.time()
        
        for step, batch in enumerate(dpo_loader):
            x_chosen = batch['x_chosen'].to(device)
            x_rejected = batch['x_rejected'].to(device)
            y_chosen = batch['y_chosen'].to(device)
            y_rejected = batch['y_rejected'].to(device)
            mask_chosen = batch['mask_chosen'].to(device)
            mask_rejected = batch['mask_rejected'].to(device)
            x = torch.cat([x_chosen, x_rejected], dim=0)
            y = torch.cat([y_chosen, y_rejected], dim=0)
            mask = torch.cat([mask_chosen, mask_rejected], dim=0)
            
            lr = get_lr(dpo_step_count, dpo_config['epochs'] * len(dpo_loader), dpo_config['learning_rate'])
            for param_group in dpo_optimizer.param_groups:
                param_group['lr'] = lr
            
            with torch.cuda.amp.autocast(dtype=torch.bfloat16 if device == 'cuda' else torch.float16):
                with torch.no_grad():
                    ref_outputs = ref_model(x)
                    ref_logits = ref_outputs.logits
                ref_log_probs = logits_to_log_probs(ref_logits, y)
                
                outputs = dpo_model(x)
                logits = outputs.logits
                policy_log_probs = logits_to_log_probs(logits, y)
                
                dpo_loss_val = dpo_loss(ref_log_probs, policy_log_probs, mask, beta=dpo_config['beta'])
                loss = (dpo_loss_val + outputs.aux_loss) / dpo_config['accumulation_steps']
            
            dpo_scaler.scale(loss).backward()
            
            if (step + 1) % dpo_config['accumulation_steps'] == 0:
                dpo_scaler.unscale_(dpo_optimizer)
                torch.nn.utils.clip_grad_norm_(dpo_model.parameters(), 1.0)
                dpo_scaler.step(dpo_optimizer)
                dpo_scaler.update()
                dpo_optimizer.zero_grad(set_to_none=True)
            
            current_loss = loss.item() * dpo_config['accumulation_steps']
            dpo_loss_history.append(current_loss)
            epoch_loss += current_loss
            dpo_step_count += 1
            
            if step % 20 == 0:
                spend_time = time.time() - start_time
                avg_loss = epoch_loss / (step + 1)
                eta_min = spend_time / max(step, 1) * (len(dpo_loader) - step) / 60
                Logger(f'Step: {step}/{len(dpo_loader)}, loss: {current_loss:.4f}, avg_loss: {avg_loss:.4f}, lr: {lr:.8f}, ETA: {eta_min:.1f}min')
        
        avg_epoch_loss = epoch_loss / len(dpo_loader)
        Logger(f"\nDPO Epoch {epoch + 1} 完成! 平均损失: {avg_epoch_loss:.4f}")
    
    # 保存模型
    ckp = f"{OUTPUT_DIR}/dpo_{inf_config.hidden_size}.pth"
    state_dict = {k: v.half().cpu() for k, v in dpo_model.state_dict().items()}
    torch.save(state_dict, ckp)
    Logger(f"\nDPO 完成! 模型保存至: {ckp}")

print("DPO 训练函数定义完成！运行 train_dpo() 开始训练。")

## 第九部分：模型推理与对话

In [ ]:
# 推理配置
inference_config = {
    'weight': 'full_sft',
    'hidden_size': 768,
    'num_hidden_layers': 8,
    'use_moe': False,
    'max_new_tokens': 512,
    'temperature': 0.85,
    'top_p': 0.95,
    'device': device,
}

# 检查权重文件
moe_suffix = '_moe' if inference_config['use_moe'] else ''
weight_path = f"{OUTPUT_DIR}/{inference_config['weight']}_{inference_config['hidden_size']}{moe_suffix}.pth"

if not os.path.exists(weight_path):
    print(f"⚠️ 权重文件不存在: {weight_path}")
    print("请先完成训练，或从 HuggingFace 下载预训练权重")
    print("\n下载预训练权重的方法：")
    print("from huggingface_hub import hf_hub_download")
    print("hf_hub_download(repo_id='jingyaogong/minimind-3-pytorch', filename='full_sft_768.pth', local_dir='./out')")
else:
    print(f"✅ 找到权重文件: {weight_path}")

print(f"\n推理配置:")
for k, v in inference_config.items():
    print(f"  {k}: {v}")

In [ ]:
# 加载推理模型
setup_seed(42)

# 创建模型配置
inf_config = MiniMindConfig(
    hidden_size=inference_config['hidden_size'],
    num_hidden_layers=inference_config['num_hidden_layers'],
    use_moe=inference_config['use_moe']
)

# 加载 Tokenizer
inf_tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

# 创建并加载模型
inf_model = MiniMindForCausalLM(inf_config)
if os.path.exists(weight_path):
    weights = torch.load(weight_path, map_location=inference_config['device'])
    inf_model.load_state_dict(weights, strict=False)
    print(f"已加载模型权重: {weight_path}")

inf_model = inf_model.eval().to(inference_config['device'])
get_model_params(inf_model, inf_config)

print("\n模型加载完成，可以开始对话！")

In [ ]:
# 单轮对话函数
def chat(prompt, history=None, open_thinking=False):
    if history is None:
        history = []
    
    # 构建对话历史
    conversation = history.copy()
    conversation.append({"role": "user", "content": prompt})
    
    # 构建输入
    if 'pretrain' in inference_config['weight']:
        inputs = inf_tokenizer.bos_token + prompt
    else:
        inputs = inf_tokenizer.apply_chat_template(
            conversation, 
            tokenize=False, 
            add_generation_prompt=True, 
            open_thinking=open_thinking
        )
    
    # 编码
    inputs = inf_tokenizer(inputs, return_tensors="pt", truncation=True).to(inference_config['device'])
    
    # 生成
    print('🧠: ', end='', flush=True)
    st = time.time()
    
    generated_ids = inf_model.generate(
        inputs=inputs["input_ids"], 
        attention_mask=inputs["attention_mask"],
        max_new_tokens=inference_config['max_new_tokens'], 
        do_sample=True, 
        pad_token_id=inf_tokenizer.pad_token_id, 
        eos_token_id=inf_tokenizer.eos_token_id,
        top_p=inference_config['top_p'], 
        temperature=inference_config['temperature'], 
        repetition_penalty=1.0
    )
    
    # 解码
    response = inf_tokenizer.decode(generated_ids[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)
    
    # 统计速度
    gen_tokens = len(generated_ids[0]) - len(inputs["input_ids"][0])
    elapsed = time.time() - st
    print(f"\n\n[Speed]: {gen_tokens / elapsed:.2f} tokens/s")
    
    return response, conversation + [{"role": "assistant", "content": response}]

print("对话函数定义完成！")

In [ ]:
# 测试对话
test_prompts = [
    "你有什么特长？",
    "为什么天空是蓝色的？",
    "请用Python写一个计算斐波那契数列的函数",
    "解释一下光合作用的基本过程",
]

print("=" * 50)
print("MiniMind 测试对话")
print("=" * 50)

history = []
for prompt in test_prompts:
    print(f"\n💬 用户: {prompt}")
    response, history = chat(prompt, history)
    print(f"🤖 模型: {response}")
    print("-" * 50)

## 第九部分：LoRA 微调（可选）

In [ ]:
# LoRA (Low-Rank Adaptation) 是一种高效的微调方法
# 它只在模型的线性层中添加低秩矩阵，大幅减少可训练参数

# 定义 LoRA 模块
class LoRA(nn.Module):
    def __init__(self, in_features, out_features, rank):
        super().__init__()
        self.rank = rank
        self.A = nn.Linear(in_features, rank, bias=False)
        self.B = nn.Linear(rank, out_features, bias=False)
        self.A.weight.data.normal_(mean=0.0, std=0.02)
        self.B.weight.data.zero_()

    def forward(self, x):
        return self.B(self.A(x))

def apply_lora(model, rank=16):
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and module.weight.shape[0] == module.weight.shape[1]:
            lora = LoRA(module.weight.shape[0], module.weight.shape[1], rank=rank).to(model.device)
            setattr(module, "lora", lora)
            original_forward = module.forward
            def forward_with_lora(x, layer1=original_forward, layer2=lora):
                return layer1(x) + layer2(x)
            module.forward = forward_with_lora

def save_lora(model, path):
    state_dict = {}
    for name, module in model.named_modules():
        if hasattr(module, 'lora'):
            lora_state = {f'{name}.lora.{k}': v.cpu().half() for k, v in module.lora.state_dict().items()}
            state_dict.update(lora_state)
    torch.save(state_dict, path)

print("LoRA 模块定义完成！")

In [ ]:
# LoRA 微调配置
lora_config = {
    'rank': 16,  # LoRA 秩
    'epochs': 5,
    'batch_size': 8,
    'learning_rate': 1e-4,
    'max_seq_len': 256,
    'accumulation_steps': 4,
    'log_interval': 20,
}

print("LoRA 配置:")
for k, v in lora_config.items():
    print(f"  {k}: {v}")

In [ ]:
# 初始化 LoRA 微调环境
setup_seed(42)

# 加载基础模型（使用 SFT 后的权重）
lora_model = MiniMindForCausalLM(inf_config)
if os.path.exists(weight_path):
    weights = torch.load(weight_path, map_location=device)
    lora_model.load_state_dict(weights, strict=False)
    print(f"已加载基础权重: {weight_path}")

# 应用 LoRA
apply_lora(lora_model, rank=lora_config['rank'])
lora_model = lora_model.to(device)

# 冻结所有参数，只训练 LoRA
for name, param in lora_model.named_parameters():
    if '.lora.' not in name:
        param.requires_grad = False

# 收集可训练参数
lora_params = [p for p in lora_model.parameters() if p.requires_grad]
print(f"\n可训练参数数量: {sum(p.numel() for p in lora_params) / 1e6:.3f}M")
print(f"总参数数量: {sum(p.numel() for p in lora_model.parameters()) / 1e6:.3f}M")

# 优化器（只优化 LoRA 参数）
lora_optimizer = optim.AdamW(lora_params, lr=lora_config['learning_rate'])
lora_scaler = torch.cuda.amp.GradScaler(enabled=True)

print("\nLoRA 微调环境初始化完成！")

In [ ]:
# LoRA 训练循环
lora_loss_history = []
lora_step_count = 0

def train_lora():
    global lora_step_count, lora_loss_history
    lora_model.train()
    
    for epoch in range(lora_config['epochs']):
        Logger(f"\n{'='*50}")
        Logger(f"LoRA Epoch [{epoch + 1}/{lora_config['epochs']}]")
        Logger(f"{'='*50}")
        
        epoch_loss = 0
        start_time = time.time()
        
        for step, (input_ids, labels) in enumerate(sft_loader):  # 复用 SFT 数据加载器
            input_ids = input_ids.to(device)
            labels = labels.to(device)
            
            lr = get_lr(lora_step_count, lora_config['epochs'] * len(sft_loader), lora_config['learning_rate'])
            for param_group in lora_optimizer.param_groups:
                param_group['lr'] = lr
            
            with torch.cuda.amp.autocast(dtype=torch.bfloat16 if device == 'cuda' else torch.float16):
                res = lora_model(input_ids, labels=labels)
                loss = (res.loss + res.aux_loss) / lora_config['accumulation_steps']
            
            lora_scaler.scale(loss).backward()
            
            if (step + 1) % lora_config['accumulation_steps'] == 0:
                lora_scaler.unscale_(lora_optimizer)
                torch.nn.utils.clip_grad_norm_(lora_params, lora_config['grad_clip'])
                lora_scaler.step(lora_optimizer)
                lora_scaler.update()
                lora_optimizer.zero_grad(set_to_none=True)
            
            current_loss = loss.item() * lora_config['accumulation_steps']
            lora_loss_history.append(current_loss)
            epoch_loss += current_loss
            lora_step_count += 1
            
            if step % lora_config['log_interval'] == 0:
                spend_time = time.time() - start_time
                avg_loss = epoch_loss / (step + 1)
                eta_min = spend_time / max(step, 1) * (len(sft_loader) - step) / 60
                Logger(f'Step: {step}/{len(sft_loader)}, loss: {current_loss:.4f}, avg_loss: {avg_loss:.4f}, lr: {lr:.8f}, ETA: {eta_min:.1f}min')
        
        avg_epoch_loss = epoch_loss / len(sft_loader)
        Logger(f"\nLoRA Epoch {epoch + 1} 完成! 平均损失: {avg_epoch_loss:.4f}")
    
    # 保存 LoRA 权重
    lora_path = f"{OUTPUT_DIR}/lora_finetune_{lora_config['rank']}.pth"
    save_lora(lora_model, lora_path)
    Logger(f"\nLoRA 微调完成! 权重保存至: {lora_path}")

print("LoRA 训练函数定义完成！运行 train_lora() 开始训练。")

## 第十部分：保存模型到 Google Drive（可选）

In [ ]:
# 将训练好的模型保存到 Google Drive
# 取消注释以下代码来启用

# from google.colab import drive
# drive.mount('/content/drive')

# import shutil
# # 创建目录
# !mkdir -p /content/drive/MyDrive/minimind-models

# # 复制模型到 Drive
# for f in os.listdir(OUTPUT_DIR):
#     if f.endswith('.pth'):
#         src = os.path.join(OUTPUT_DIR, f)
#         dst = f'/content/drive/MyDrive/minimind-models/{f}'
#         shutil.copy(src, dst)
#         print(f"已保存: {dst}")

print("如需保存到 Google Drive，请取消注释上方代码")

## 附录：快速开始指南

In [ ]:
# 快速开始：一键完成预训练 → SFT → 推理
def quick_start():
    print("=" * 50)
    print("MiniMind Colab 快速开始")
    print("=" * 50)
    
    print("\n📚 步骤 1: 下载数据")
    print("运行数据下载单元格")
    
    print("\n📚 步骤 2: 预训练")
    print("运行: train_pretrain()")
    
    print("\n📝 步骤 3: 指令微调")
    print("运行: train_sft()")
    
    print("\n💬 步骤 4: 对话推理")
    print("运行测试对话单元格")
    
    print("\n" + "=" * 50)
    print("提示: Colab 免费 T4 GPU 约需 1-2 小时完成全部训练")

quick_start()

## 常见问题

### Q1: CUDA Out of Memory 怎么办？
- 减小 `batch_size`
- 减小 `max_seq_len`
- 增加 `accumulation_steps`

### Q2: 训练太慢怎么办？
- 确保使用了 GPU 运行时 (Runtime > Change runtime type > GPU)
- 使用 Colab Pro 可获得更好的 GPU (A100)

### Q3: 数据下载失败怎么办？
- 手动从 [HuggingFace](https://huggingface.co/datasets/jingyaogong/minimind_dataset) 下载
- 上传到 Colab 文件浏览器

### Q4: 如何保存模型避免丢失？
- Colab 会话结束后文件会被清除
- 使用第九部分的代码保存到 Google Drive

---

**项目信息**
- 原始项目: [minimind](https://github.com/jingyaogong/minimind)
- 许可证: Apache 2.0
- 作者: Jingyao Gong

**引用**
```bibtex
@misc{minimind,
  title = {MiniMind: Train a Tiny LLM from Scratch},
  author = {Jingyao Gong},
  year = {2024},
  url = {https://github.com/jingyaogong/minimind},
}
```